# B2B Payment Risk & Collection Decision Engine — Full Pipeline Walkthrough

This notebook walks the entire project end to end: raw CSV in, ranked collection queue out.

**What it covers**

| # | Stage |
|---|---|
| 1–2 | Load raw data, audit quality, split open vs closed |
| 3–4 | Construct the target, exploratory analysis |
| 5 | Feature engineering — **including the target leakage that was found and fixed** |
| 6–7 | Build the feature matrix, feature selection |
| 8–9 | Chronological splitting, training the three models |
| 10–11 | Evaluation, calibration, feature importance |
| 12 | Leakage experiment: what the leak was actually worth |
| 13 | **Decision engine — expected value, tiers, capacity-constrained ranking** |
| 14 | Strategy comparison |

**Before you run:** ~3–5 minutes end to end. The heaviest cells are model training (§9, three gradient-boosting fits), permutation importance (§11), and the leakage experiment (§12, trains a fourth model). Everything else is seconds.

**Central idea:** the model is not the product. The product is *which invoices a capacity-limited collections team should call today*. Accuracy is an input to that decision, not the goal.

## 0. Setup

The notebook reuses the real project modules in `deploy/src/` rather than reimplementing them, so
what you see here is exactly what the Flask app runs. The one exception is the *leaky* feature
builder in §5, which is reconstructed deliberately to show what was wrong.

In [ ]:
# Picks up edits to deploy/src/*.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

# Walk up until we find the deploy/ folder, so this runs from anywhere in the repo.
ROOT = Path.cwd().resolve()
while not (ROOT / "deploy").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DEPLOY = ROOT / "deploy"

# Fail loudly here rather than with a confusing error ten cells later. This happens
# when the kernel was started outside the repo, or the notebook was moved out of it.
if not (DEPLOY / "dataset.csv").exists():
    raise FileNotFoundError(
        f"Could not locate deploy/dataset.csv by walking up from {Path.cwd()}.\n"
        "Start Jupyter from inside the project, or set DEPLOY manually to the deploy/ folder."
    )

sys.path.insert(0, str(DEPLOY))

print("project root :", ROOT)
print("deploy dir   :", DEPLOY)
print("dataset      :", (DEPLOY / "dataset.csv").exists())

## 1. Load the raw data

In [ ]:
raw = pd.read_csv(DEPLOY / "dataset.csv")
print("shape:", raw.shape)
raw.head()

In [ ]:
pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "nulls": raw.isna().sum(),
    "null_%": (raw.isna().mean() * 100).round(2),
    "n_unique": raw.nunique(),
})

### The columns that matter

| column | role |
|---|---|
| `posting_date` | when the invoice was raised — **the moment of prediction** |
| `due_in_date` | when payment is contractually due |
| `clear_date` | when it was actually paid. Null for open invoices. **Source of the target** |
| `total_open_amount` | invoice value (USD) |
| `cust_number` | customer key — drives all history features |
| `cust_payment_terms` | contractual terms code (e.g. NAA8) |
| `business_code` | business unit |
| `isOpen` | 1 = unpaid (predict these), 0 = paid (train on these) |

Everything else (`doc_id`, `area_business`, `invoice_currency`, the duplicated
`document_create_date.1`, …) is either an identifier, constant, or redundant, and is not used.

In [ ]:
raw["posting_date"] = pd.to_datetime(raw["posting_date"], format="mixed")
raw["due_in_date"]  = pd.to_datetime(raw["due_in_date"].astype(str), format="%Y%m%d")
raw["clear_date"]   = pd.to_datetime(raw["clear_date"], format="mixed", errors="coerce")

print("rows                :", len(raw))
print("unique invoice_id   :", raw.invoice_id.nunique(), " -> duplicates:", len(raw) - raw.invoice_id.nunique())
print("unique customers    :", raw.cust_number.nunique())
print("payment terms codes :", raw.cust_payment_terms.nunique())
print("business codes      :", raw.business_code.nunique())

Note the duplicate `invoice_id` values. They are a real data-quality issue worth knowing about —
in the Databricks migration this is exactly what the Silver layer's deduplication step exists to
handle. Here they are left alone because the model trains per-row, not per-invoice.

## 2. Open vs closed — and why their order matters

In [ ]:
closed = raw[raw.isOpen == 0].copy()
open_inv = raw[raw.isOpen == 1].copy()

for name, d in [("closed (trainable)", closed), ("open (to predict)", open_inv)]:
    print(f"{name:<20} n={len(d):>6}  posting {d.posting_date.min().date()} .. {d.posting_date.max().date()}")
print()
print("closed clear_date    :", closed.clear_date.min().date(), "..", closed.clear_date.max().date())
print("closed rows missing clear_date:", closed.clear_date.isna().sum())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
ax.hist(closed.posting_date, bins=60, label=f"closed (n={len(closed):,})", alpha=.85)
ax.hist(open_inv.posting_date, bins=25, label=f"open (n={len(open_inv):,})", alpha=.85)
ax.axvline(open_inv.posting_date.min(), ls="--", c="k", lw=1)
ax.set_title("Invoices by posting date — the two sets are sequential, not interleaved")
ax.set_ylabel("invoices"); ax.legend()
plt.tight_layout(); plt.show()

**This single chart drives the whole modelling strategy.**

Closed invoices run to 2020-02-27; open invoices start there and run to 2020-05-22. They do not
overlap. The real task is therefore *train on the past, predict the future* — so any evaluation
using a random shuffle is measuring the wrong thing. We return to this in §8.

## 3. Constructing the target

In [ ]:
closed["days_late"] = (closed.clear_date - closed.due_in_date).dt.days
closed = closed.dropna(subset=["days_late", "clear_date"]).reset_index(drop=True)

print(closed.days_late.describe().round(2).to_string())
print()
print(f"late (>0)        : {(closed.days_late > 0).mean()*100:.1f}%")
print(f"on time / early  : {(closed.days_late <= 0).mean()*100:.1f}%")
print(f"very late (>30d) : {(closed.days_late > 30).mean()*100:.1f}%")

`days_late = clear_date - due_in_date`

Framing the target *relative to the due date* rather than as a raw calendar date is deliberate:
it makes the target comparable across invoices with different terms, and it is the quantity the
business actually cares about. A predicted payment date is recovered at the end with
`due_in_date + days_late`.

Note the target is **signed** — negative means paid early. Roughly 58% of invoices are paid on
time or early, so a model that only predicts lateness would be wrong most of the time.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
a1.hist(closed.days_late.clip(-40, 60), bins=100)
a1.axvline(0, c="k", ls="--", lw=1); a1.set_title("days_late (clipped to [-40, 60])"); a1.set_xlabel("days")
a2.hist(closed.days_late, bins=200, log=True)
a2.axvline(0, c="k", ls="--", lw=1); a2.set_title("full range, log count — note the long right tail")
plt.tight_layout(); plt.show()

print("skew:", round(float(closed.days_late.skew()), 2), " kurtosis:", round(float(closed.days_late.kurt()), 2))

The distribution is sharply peaked near zero with a long right tail out to +204 days. That shape
matters later: **squared-error loss will be dragged around by the tail**, which is one reason the
±3-day hit rate and the prediction interval are reported alongside MAE.

## 4. Exploratory analysis

In [ ]:
terms = (closed.groupby("cust_payment_terms")
         .agg(n=("days_late", "size"), avg_late=("days_late", "mean"), late_rate=("days_late", lambda s: (s > 0).mean()))
         .query("n >= 200").sort_values("avg_late", ascending=False))

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(terms.index.astype(str), terms.avg_late)
ax.set_title("Mean days late by payment terms (codes with >= 200 invoices)")
ax.set_ylabel("mean days late"); ax.axhline(0, c="k", lw=1); plt.xticks(rotation=90)
plt.tight_layout(); plt.show()
terms.round(2)

Payment terms separate behaviour strongly — and importantly this is a feature known *at invoice
creation*, with no leakage risk. That is why it ends up as the second most important feature after
the leak is removed.

In [ ]:
biz = (closed.groupby("business_code")
       .agg(n=("days_late", "size"), avg_late=("days_late", "mean"),
            late_rate=("days_late", lambda s: (s > 0).mean()), avg_amount=("total_open_amount", "mean")))
biz.round(2)

In [ ]:
closed["amount_decile"] = pd.qcut(closed.total_open_amount, 10, labels=False, duplicates="drop")
dec = closed.groupby("amount_decile").agg(avg_late=("days_late", "mean"),
                                          late_rate=("days_late", lambda s: (s > 0).mean()),
                                          avg_amount=("total_open_amount", "mean"))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4))
a1.plot(dec.index, dec.avg_late, marker="o"); a1.set_title("Mean days late by invoice-amount decile")
a1.set_xlabel("amount decile"); a1.axhline(0, c="k", lw=1)
a2.plot(dec.index, dec.late_rate, marker="o", color="tab:orange"); a2.set_title("Late rate by amount decile")
a2.set_xlabel("amount decile"); a2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout(); plt.show()

Amount has a mild relationship with lateness — larger invoices are somewhat later. It is a weak
predictor on its own, but it matters enormously in the decision engine, because expected value
scales linearly with amount. **A weak feature can still be a decisive input to a decision.**

In [ ]:
cust = closed.groupby("cust_number").agg(n=("days_late", "size"), avg_late=("days_late", "mean"),
                                         std_late=("days_late", "std"))
print("customers:", len(cust))
print(cust.n.describe().round(1).to_string())
print()
for t in (1, 2, 5, 10):
    share = closed.cust_number.isin(cust[cust.n <= t].index).mean()
    print(f"customers with <= {t:>2} invoices: {(cust.n <= t).sum():>5}  covering {share*100:>5.1f}% of rows")

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.hist(np.log10(cust.n), bins=50)
ax.set_title("Customer invoice counts (log10) — a few huge accounts, a long tail of tiny ones")
ax.set_xlabel("log10(invoice count)"); plt.tight_layout(); plt.show()

**Read this carefully — it is the setup for the leakage section.**

Half of all customers have 3 or fewer invoices. Any feature built by averaging a customer's
historical lateness is, for those customers, computed from almost no data — and if that average
includes the invoice's *own* outcome, the feature is partly a copy of the label.

In [ ]:
monthly = closed.set_index("posting_date").resample("ME").agg(
    avg_late=("days_late", "mean"), n=("days_late", "size"))
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(monthly.index, monthly.avg_late, marker="o")
ax.axhline(0, c="k", lw=1); ax.set_title("Mean days late over time (seasonality / drift check)")
ax.set_ylabel("mean days late"); plt.tight_layout(); plt.show()

## 5. Feature engineering — and the leakage that was hiding here

This is the most important section of the notebook.

The intuition behind the strongest features is sound: *a customer who has paid late before will
probably pay late again*. The question is *how* you compute that history.

### 5a. The original (leaky) approach

The first version of this project built one row per customer, aggregating **all closed invoices at
once**, then joined it onto every invoice:

In [ ]:
def build_history_LEAKY(src):
    # Reconstruction of the original approach. Kept only for contrast - do not use.
    g = src.groupby("cust_number")
    return pd.DataFrame({
        "cust_avg_days_late": g.days_late.mean(),
        "cust_std_days_late": g.days_late.std(),
        "cust_invoice_count": g.days_late.count(),
        "cust_min_late":      g.days_late.min(),
        "cust_max_late":      g.days_late.max(),
        "cust_avg_amount":    g.total_open_amount.mean(),
    }).reset_index()

leaky_hist = build_history_LEAKY(closed)
leaky_hist.head()

### Why this leaks — two separate problems

**Problem 1 — an invoice sees its own outcome.** `cust_avg_days_late` for a customer includes the
very row it is attached to. For a customer with one invoice, the feature *equals the label*:

In [ ]:
single = cust[cust.n == 1].index[:5]
demo = closed[closed.cust_number.isin(single)][["cust_number", "days_late"]].merge(
    leaky_hist[["cust_number", "cust_avg_days_late", "cust_invoice_count"]], on="cust_number")
print("Customers with exactly one closed invoice:\n")
print(demo.to_string(index=False))
print("\nfeature == label for every row above:", np.allclose(demo.days_late, demo.cust_avg_days_late))
print(f"\n{(cust.n == 1).sum()} customers are in this position.")

**Problem 2 — every invoice sees the future.** Even for a customer with 500 invoices, the average
is computed over their *entire* history including invoices that clear months later. At the moment
you actually predict, those outcomes do not exist yet.

The two problems compound: the feature is part label, part crystal ball.

### 5b. The fix — point-in-time features

The rule: **when predicting an invoice raised on date `t`, you may only use outcomes that had
already cleared before `t`.**

Not *posted* before `t` — *cleared* before `t`. An invoice raised earlier but still unpaid tells
you nothing about how late it will be.

Implementation: build a running per-customer aggregate stamped with `clear_date`, then join it
as-of each invoice's `posting_date` with a backward `merge_asof`.

In [ ]:
from src.features import (FEATURE_COLS, HIST_COLS, add_features,
                          build_customer_timeline, latest_customer_state)

timeline = build_customer_timeline(closed)
print("timeline rows:", len(timeline), "(one per cleared invoice, not one per customer)")
timeline.head()

In [ ]:
# Follow one busy customer through time.
candidates = cust[(cust.n >= 6) & (cust.n <= 12)].index
busy = candidates[0] if len(candidates) else cust.n.idxmax()
view = timeline[timeline.cust_number == busy].head(8)
print(f"Customer {busy} - what was known after each of their invoices cleared:\n")
print(view[["clear_date", "cust_invoice_count", "cust_avg_days_late", "cust_min_late", "cust_max_late"]]
      .round(2).to_string(index=False))

Each row is the state of knowledge *immediately after* that invoice cleared. `cust_invoice_count`
grows by one each time; `cust_avg_days_late` is a running mean. Joining as-of `posting_date` picks
the last row that had already happened.

Two details in `add_features` that are easy to get wrong:

- `direction="backward"` — take the most recent state at or before the invoice date, never a later one.
- `allow_exact_matches=False` — an invoice raised the *same day* another clears must not see it.
  Same-day visibility is a subtle leak that a date-only join would let through.

In [ ]:
# Side by side: what the leaky build gives this customer vs. what point-in-time gives.
from src.train import build_encoders, build_defaults

enc_demo = build_encoders(closed)
def_demo = build_defaults(closed)
rows = closed[closed.cust_number == busy].sort_values("posting_date")
pit = add_features(rows, enc_demo, timeline, def_demo)
lk  = rows.merge(leaky_hist, on="cust_number", how="left")

cmp = pd.DataFrame({
    "posting_date":     rows.posting_date.dt.date.values,
    "actual_days_late": rows.days_late.values,
    "LEAKY_avg":        lk.cust_avg_days_late.values.round(2),
    "PIT_avg":          pit.cust_avg_days_late.values.round(2),
    "PIT_count":        pit.cust_invoice_count.values.astype(int),
})
print(f"Customer {busy}:\n")
print(cmp.to_string(index=False))

The `LEAKY_avg` column is **constant** — the same number on every row, computed once from all of
this customer's invoices including their futures. `PIT_avg` changes as evidence accumulates, and
starts at the global default when nothing is known yet.

That constant column is what the model was really learning from.

## 6. Building the feature matrix

In [ ]:
from src.train import load_closed, chronological_split, GBM

print("Model features (%d):" % len(FEATURE_COLS))
for f in FEATURE_COLS:
    tag = "  <- derived from the target (needs point-in-time care)" if f in HIST_COLS else ""
    print(f"  - {f}{tag}")

| group | features | leakage risk |
|---|---|---|
| Amount | `amount_log` | none — known at creation |
| Temporal | `month`, `day_of_week`, `is_month_end`, `is_year_end` | none |
| Contractual | `payment_terms_encoded`, `business_code_encoded` | none |
| Customer history | the six `cust_*` columns | **high** — these are aggregates of the label |

`amount_log` uses `log1p(|amount|)` because the raw amount is heavily right-skewed and contains
occasional negatives (credits).

## 7. Feature selection

With only 13 candidate features and a tree model that handles irrelevant inputs gracefully,
aggressive selection is not needed. The useful questions are *which features are redundant* and
*which carry the signal*.

In [ ]:
enc = build_encoders(closed)
dfl = build_defaults(closed)
featured_all = add_features(closed, enc, timeline, dfl)

corr = featured_all[FEATURE_COLS + ["days_late"]].corr()
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6.5)
ax.set_title("Feature correlation (point-in-time features)")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

In [ ]:
tc = corr["days_late"].drop("days_late").sort_values(key=abs, ascending=False)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.barh(tc.index[::-1], tc.values[::-1])
ax.axvline(0, c="k", lw=1); ax.set_title("Linear correlation with days_late")
plt.tight_layout(); plt.show()
tc.round(3).to_frame("corr_with_target")

Correlations are modest across the board — no single feature comes close to determining the
target. That is *healthy*. If you ever see a feature correlating 0.9 with the target here, assume
leakage before assuming brilliance.

`cust_min_late` / `cust_max_late` / `cust_avg_days_late` are correlated with each other, which is
expected. Tree ensembles tolerate this; it only distorts the *attribution* of importance between
them, not predictive quality.

## 8. Splitting — chronological, not random

The natural instinct is `train_test_split(shuffle=True)`. Here that would be a mistake, for the
reason established in §2: the deployment task is strictly forward in time. A shuffled split lets
the model learn from November to predict January, which it will never be able to do in production.

In [ ]:
train, test = chronological_split(closed)
print(f"train: {len(train):>6}  {train.posting_date.min().date()} .. {train.posting_date.max().date()}")
print(f"test : {len(test):>6}  {test.posting_date.min().date()} .. {test.posting_date.max().date()}")
print("\nno overlap:", train.posting_date.max() <= test.posting_date.min())

In [ ]:
# CRITICAL: the timeline must be built from the TRAIN fold only.
# Building it from `closed` would let test outcomes flow into training features -
# the exact leak we just removed, reintroduced through the back door.
enc_tr  = build_encoders(train)
def_tr  = build_defaults(train)
tl_tr   = build_customer_timeline(train)

f_train = add_features(train, enc_tr, tl_tr, def_tr)
f_test  = add_features(test,  enc_tr, tl_tr, def_tr)

X_train, y_train = f_train[FEATURE_COLS].values, f_train.days_late.values
X_test,  y_test  = f_test[FEATURE_COLS].values,  f_test.days_late.values
print("X_train", X_train.shape, " X_test", X_test.shape)
print("cold-start rows in test (no prior cleared history):", int((f_test.cust_invoice_count == 0).sum()))

## 9. Model training

Three gradient-boosting regressors, identical hyperparameters, different loss functions:

| model | loss | output |
|---|---|---|
| `model_mean` | squared error | point estimate (P50-ish) |
| `model_lower` | quantile, `alpha=0.1` | P10 — optimistic bound |
| `model_upper` | quantile, `alpha=0.9` | P90 — pessimistic bound |

The pair of quantile models produces an **80% prediction interval**. This is not decoration: the
decision engine derives its probability of lateness directly from the interval, so interval
*calibration* feeds straight into the ranking. A miscalibrated interval produces a confidently
wrong call queue.

This cell is the slowest in the notebook — roughly 30–60 seconds.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

print("hyperparameters:", GBM)
model_mean  = GradientBoostingRegressor(**GBM).fit(X_train, y_train)
model_lower = GradientBoostingRegressor(loss="quantile", alpha=0.1, **GBM).fit(X_train, y_train)
model_upper = GradientBoostingRegressor(loss="quantile", alpha=0.9, **GBM).fit(X_train, y_train)

pred  = model_mean.predict(X_test)
lower = model_lower.predict(X_test)
upper = model_upper.predict(X_test)
print("done.")

## 10. Evaluation

In [ ]:
def report(y, p, lo, hi, label=""):
    m = {
        "MAE":        float(np.mean(np.abs(y - p))),
        "RMSE":       float(np.sqrt(np.mean((y - p) ** 2))),
        "median_AE":  float(np.median(np.abs(y - p))),
        "within_1d":  float(np.mean(np.abs(y - p) <= 1) * 100),
        "within_3d":  float(np.mean(np.abs(y - p) <= 3) * 100),
        "within_7d":  float(np.mean(np.abs(y - p) <= 7) * 100),
        "bias":       float(np.mean(p - y)),
        "PI_coverage": float(np.mean((y >= lo) & (y <= hi)) * 100),
        "PI_width":   float(np.mean(hi - lo)),
    }
    print(f"=== {label} ===")
    for k, v in m.items():
        print(f"  {k:<12}{v:>8.2f}")
    return m

metrics = report(y_test, pred, lower, upper, "Chronological hold-out (n=%d)" % len(y_test))

**How to read these.**

- **MAE ~3.2 days** — the typical miss. Given a target with std 10.8, that is real signal.
- **PI coverage** should be near 80% by construction. Meaningfully below means the intervals are
  too narrow and the engine will be overconfident.
- **bias** near zero means the model is not systematically early or late.
- **within_3d** is the business-legible metric: how often the prediction lands inside a workable window.

A useful sanity baseline: predicting the training mean for every invoice.

In [ ]:
naive = np.full_like(y_test, y_train.mean(), dtype=float)
print(f"naive (predict train mean = {y_train.mean():.2f}):  MAE {np.mean(np.abs(y_test-naive)):.2f}   "
      f"RMSE {np.sqrt(np.mean((y_test-naive)**2)):.2f}")
print(f"model                              :  MAE {metrics['MAE']:.2f}   RMSE {metrics['RMSE']:.2f}")
print(f"\nMAE improvement over naive: {(1 - metrics['MAE']/np.mean(np.abs(y_test-naive)))*100:.1f}%")

In [ ]:
resid = pred - y_test
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].scatter(y_test, pred, s=3, alpha=.15)
lim = [-40, 60]; axes[0].plot(lim, lim, "k--", lw=1); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("actual"); axes[0].set_ylabel("predicted"); axes[0].set_title("Predicted vs actual")
axes[1].hist(resid.clip(-40, 40), bins=80); axes[1].axvline(0, c="k", ls="--", lw=1)
axes[1].set_title("Residuals (pred - actual)"); axes[1].set_xlabel("days")
axes[2].scatter(pred, resid, s=3, alpha=.15); axes[2].axhline(0, c="k", ls="--", lw=1)
axes[2].set_xlim(-20, 30); axes[2].set_ylim(-50, 50)
axes[2].set_xlabel("predicted"); axes[2].set_ylabel("residual"); axes[2].set_title("Residual vs prediction")
plt.tight_layout(); plt.show()

The predicted-vs-actual cloud is compressed toward the centre — the model under-predicts extreme
lateness. That is expected from squared-error boosting on a long-tailed target, and it is precisely
why the P90 model exists: the *interval* carries the tail risk that the point estimate flattens.

In [ ]:
# Is the 80% interval actually 80%? Check coverage across interval widths.
w = upper - lower
inside = (y_test >= lower) & (y_test <= upper)
buckets = pd.qcut(w, 6, duplicates="drop")
cov = pd.DataFrame({"width": w, "inside": inside}).groupby(buckets, observed=True).agg(
    n=("inside", "size"), coverage=("inside", "mean"), mean_width=("width", "mean"))
cov["coverage"] = (cov.coverage * 100).round(1)
print(cov.round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(range(len(cov)), cov.coverage)
ax.axhline(80, c="r", ls="--", lw=1.2, label="nominal 80%")
ax.set_xticks(range(len(cov))); ax.set_xticklabels([f"{v:.1f}d" for v in cov.mean_width], rotation=0)
ax.set_xlabel("mean interval width"); ax.set_ylabel("coverage %")
ax.set_title("Calibration: coverage by interval width"); ax.legend()
plt.tight_layout(); plt.show()

Ideally every bar sits on the red line. Bars **below** 80% on narrow intervals mean the model is
overconfident exactly where it claims most certainty — the dangerous failure mode, because those
are the invoices the engine will rank most aggressively.

In [ ]:
# Where does the model do badly? Error by customer-history depth and by amount.
diag = pd.DataFrame({
    "abs_err": np.abs(resid),
    "hist_depth": pd.cut(f_test.cust_invoice_count, [-1, 0, 2, 10, 50, 1e9],
                         labels=["0 (cold)", "1-2", "3-10", "11-50", "50+"]),
    "amount_bucket": pd.qcut(f_test.total_open_amount, 5, labels=["Q1 low", "Q2", "Q3", "Q4", "Q5 high"]),
})
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4))
diag.groupby("hist_depth", observed=True).abs_err.mean().plot(kind="bar", ax=a1, rot=0)
a1.set_title("MAE by customer history depth"); a1.set_ylabel("MAE (days)")
diag.groupby("amount_bucket", observed=True).abs_err.mean().plot(kind="bar", ax=a2, rot=0, color="tab:orange")
a2.set_title("MAE by invoice amount"); a2.set_ylabel("MAE (days)")
plt.tight_layout(); plt.show()
print(diag.groupby("hist_depth", observed=True).agg(n=("abs_err", "size"), MAE=("abs_err", "mean")).round(2).to_string())

Cold-start invoices — customers with no cleared history at prediction time — are the weakest
segment. Under the old leaky pipeline this was invisible, because those rows were quietly handed a
customer average built from their own future. Making the weakness *visible* is a feature of the
fix, not a regression.

## 11. Feature importance

In [ ]:
imp = pd.Series(model_mean.feature_importances_, index=FEATURE_COLS).sort_values()
colors = ["tab:red" if f in HIST_COLS else "tab:blue" for f in imp.index]
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(imp.index, imp.values * 100, color=colors)
ax.set_xlabel("importance %"); ax.set_title("Impurity importance (red = target-derived history)")
plt.tight_layout(); plt.show()
print(f"share carried by customer-history features: {imp[HIST_COLS].sum()*100:.1f}%")

Impurity importance is biased toward high-cardinality continuous features, so confirm with
**permutation importance**, which measures the actual degradation when a feature is shuffled.
This cell takes ~30 seconds.

In [ ]:
from sklearn.inspection import permutation_importance

sub = min(4000, len(X_test))
pi = permutation_importance(model_mean, X_test[:sub], y_test[:sub],
                            n_repeats=5, random_state=42, scoring="neg_mean_absolute_error")
pis = pd.Series(pi.importances_mean, index=FEATURE_COLS).sort_values()
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(pis.index, pis.values, xerr=pd.Series(pi.importances_std, index=FEATURE_COLS)[pis.index],
        color=["tab:red" if f in HIST_COLS else "tab:blue" for f in pis.index])
ax.set_xlabel("MAE increase when shuffled (days)"); ax.set_title("Permutation importance")
plt.tight_layout(); plt.show()
pis.sort_values(ascending=False).round(4).to_frame("mae_increase")

## 12. What was the leak actually worth?

Rather than argue about it, measure it. Train the same model on leaky features with a random split
— the original setup — and compare against the clean chronological pipeline.

This cell trains a fourth model; ~20 seconds.

In [ ]:
from sklearn.model_selection import train_test_split

# Variant A: leaky features + random split (the original setup).
leaky_featured = closed.merge(leaky_hist, on="cust_number", how="left")
for c in HIST_COLS:
    leaky_featured[c] = leaky_featured[c].fillna(dfl[c])
tmp = add_features(closed, enc, timeline, dfl)          # for the non-history columns
for c in FEATURE_COLS:
    if c not in HIST_COLS:
        leaky_featured[c] = tmp[c].values

XA, yA = leaky_featured[FEATURE_COLS].values, leaky_featured.days_late.values
XA_tr, XA_te, yA_tr, yA_te = train_test_split(XA, yA, test_size=0.2, random_state=42)
mA = GradientBoostingRegressor(**GBM).fit(XA_tr, yA_tr)
pA = mA.predict(XA_te)

comparison = pd.DataFrame([
    {"pipeline": "A. leaky features + random split", "MAE": np.mean(np.abs(yA_te - pA)),
     "RMSE": np.sqrt(np.mean((yA_te - pA) ** 2)), "within_3d": np.mean(np.abs(yA_te - pA) <= 3) * 100,
     "history_importance_%": sum(mA.feature_importances_[FEATURE_COLS.index(c)] for c in HIST_COLS) * 100},
    {"pipeline": "B. point-in-time + chronological", "MAE": metrics["MAE"],
     "RMSE": metrics["RMSE"], "within_3d": metrics["within_3d"],
     "history_importance_%": imp[HIST_COLS].sum() * 100},
]).set_index("pipeline").round(2)
comparison

The leaky pipeline reports a **better** MAE. It is not a better model — it is the same model
grading its own homework. The honest number is the point-in-time one, and the gap is what you
would have discovered in production, after promising the lower figure to stakeholders.

Notice also how much more of the model's importance sat in target-derived features under A.

**This is the single most valuable story in the project.** Finding, measuring, and fixing leakage
in your own pipeline demonstrates more maturity than any accuracy number.

## 13. The decision engine

Everything so far produced a *prediction*. Nothing so far produced a *decision*.

A collections team can make roughly 20 calls a day against 10,000 open invoices. Ranking by
predicted lateness is wrong — a 40-day-late $500 invoice matters less than a 4-day-late $2M one.
Ranking by amount is also wrong — the biggest invoice may be from a customer who always pays on
time and never responds to calls.

The engine ranks by **expected monetary value of making the call**.

In [ ]:
from src import decision
from src.decision import (_p_late, _p_responds, expected_value, _assign_tier,
                          build_priority_table, DAILY_CAPACITY, CALL_COST,
                          DAYS_ACCELERATED, DAILY_CAPITAL_RATE)

for k in ["DAILY_CAPITAL_RATE", "CALL_COST", "DAYS_ACCELERATED", "DAILY_CAPACITY",
          "CALL_DELAY_DAYS", "CALL_AMOUNT_USD", "REMIND_DELAY_DAYS"]:
    print(f"  {k:<20} = {getattr(decision, k)}")

### 13a. The expected value formula

$$
EV = \underbrace{P(\text{late}) \times P(\text{responds})}_{\text{chance the call achieves anything}}
     \times \underbrace{d_{\text{accel}} \times r_{\text{daily}} \times \text{amount}}_{\text{cash value of paying } d \text{ days sooner}}
     \;-\; \underbrace{c_{\text{call}}}_{\text{cost of the call}}
$$

Each term, and where it comes from:

| term | value | source |
|---|---|---|
| $P(\text{late})$ | 0–1 | derived from the **P10/P90 interval** — this is why calibration matters |
| $P(\text{responds})$ | 0.2–0.8 | inverse of the customer's payment variability |
| $d_{accel}$ | 3 days | assumed acceleration from a successful call |
| $r_{daily}$ | 0.0003 | daily cost of capital (~10%/yr) |
| $c_{call}$ | \$15 | loaded cost of one collector call |

**A positive EV means the call is worth making.** Because the call costs $15 regardless, small
invoices are structurally unprofitable to chase — the engine discovers this rather than being told.

In [ ]:
# P(late) as a function of the prediction interval.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4))
xs = np.linspace(-10, 15, 200)
a1.plot(xs, [_p_late(x - 3, x + 3) for x in xs], label="width 6d")
a1.plot(xs, [_p_late(x - 8, x + 8) for x in xs], label="width 16d")
a1.set_xlabel("interval centre (days late)"); a1.set_ylabel("P(late)")
a1.set_title("P(late) from the prediction interval"); a1.legend()
stds = np.linspace(0, 30, 200)
a2.plot(stds, [_p_responds(s) for s in stds], color="tab:orange")
a2.set_xlabel("customer std of days late"); a2.set_ylabel("P(responds)")
a2.set_title("P(responds) - erratic payers respond less predictably")
plt.tight_layout(); plt.show()

`_p_late` is a deliberately simple heuristic over the interval: an interval entirely below zero
returns 0.05, one starting above +5 days returns 0.95, and in between it interpolates by where zero
falls inside the interval. `_p_responds` clips to [0.2, 0.8] — never certainty in either direction.

Both are **assumptions, not estimates**. A stronger version would learn `P(responds)` from actual
intervention outcomes; there is no such data here, and the notebook should not pretend otherwise.

### 13b. A worked example — one invoice, start to finish

In [ ]:
ex_amount, ex_lower, ex_upper, ex_std = 250_000.0, -2.0, 7.0, 6.0

p_l = _p_late(ex_lower, ex_upper)
p_r = _p_responds(ex_std)
benefit = p_l * p_r * DAYS_ACCELERATED * DAILY_CAPITAL_RATE * ex_amount
ev = expected_value(ex_amount, ex_lower, ex_upper, ex_std)

print(f"Invoice: ${ex_amount:,.0f}, predicted {ex_lower} to {ex_upper} days late, customer std {ex_std}\n")
print(f"  1. P(late)      = upper/(upper-lower) = {ex_upper}/{ex_upper-ex_lower} -> {p_l:.3f}")
print(f"  2. P(responds)  = 1 - {ex_std}/20, clipped to [0.2,0.8] -> {p_r:.3f}")
print(f"  3. cash value   = {DAYS_ACCELERATED} days x {DAILY_CAPITAL_RATE} x ${ex_amount:,.0f} "
      f"= ${DAYS_ACCELERATED*DAILY_CAPITAL_RATE*ex_amount:,.2f}")
print(f"  4. x P(late) x P(responds)            = ${benefit:,.2f}")
print(f"  5. minus call cost ${CALL_COST:.0f}                 = ${ev:,.2f}")
print(f"\n  EV = ${ev:,.2f}  -> {'WORTH CALLING' if ev > 0 else 'not worth calling'}")

In [ ]:
# The break-even invoice size, holding the probabilities fixed.
sizes = np.logspace(3, 6.5, 200)
evs = [expected_value(s, ex_lower, ex_upper, ex_std) for s in sizes]
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.semilogx(sizes, evs); ax.axhline(0, c="r", ls="--", lw=1.2)
be = sizes[np.argmin(np.abs(np.array(evs)))]
ax.axvline(be, c="k", ls=":", lw=1)
ax.set_xlabel("invoice amount ($, log)"); ax.set_ylabel("expected value ($)")
ax.set_title(f"EV vs invoice size - break-even near ${be:,.0f}")
plt.tight_layout(); plt.show()

### 13c. Action tiers

EV decides *ordering*. A separate tier rule decides *what kind of contact* is appropriate, using
predicted delay and invoice size:

| tier | rule |
|---|---|
| `CALL` | delay > 7 days (any amount), **or** delay ≥ 3 days and amount > \$50K |
| `REMIND` | delay ≥ 3 days, amount ≤ \$50K — an email is proportionate |
| `WATCH` | delay 1–2 days — monitor, do not contact |
| `OK` | predicted on time or early |

In [ ]:
grid = [(d, a) for d in (0, 1, 3, 5, 8, 15) for a in (5_000, 60_000)]
print(f"{'delay':>6}{'amount':>12}   tier")
for d, a in grid:
    print(f"{d:>6}{a:>12,}   {_assign_tier(d, a)}")

### 13d. The capacity constraint — where it becomes decision science

This is the part that separates the project from a prediction demo.

Many more invoices qualify as `CALL` than the team can handle. So the engine sorts qualifying
invoices by EV, keeps the top `DAILY_CAPACITY` (20) as `CALL`, and **demotes the remainder to
`REMIND`**. The queue is not "everything urgent" — it is "the 20 calls with the highest expected
return, and a cheaper channel for the rest."

The objective is not accuracy. It is *expected value delivered under a hard constraint*.

In [ ]:
from src.predict import predict

# Score the real open book with the models trained in this notebook.
serve_models = {"model_mean": model_mean, "model_lower": model_lower, "model_upper": model_upper,
                "label_encoders": enc, "customer_timeline": timeline, "global_defaults": dfl}
open_scored = predict(open_inv.reset_index(drop=True), serve_models)
print("scored open invoices:", len(open_scored))
print(open_scored[["days_late_pred", "days_late_lower", "days_late_upper"]].describe().round(2).to_string())

In [ ]:
priority = build_priority_table(open_scored)
print(priority.action.value_counts().to_string())
print(f"\nCALL capped at DAILY_CAPACITY = {DAILY_CAPACITY}: {(priority.action=='CALL').sum() == DAILY_CAPACITY}")
print(f"total EV of the CALL queue: ${priority[priority.action=='CALL'].expected_value.sum():,.2f}")

In [ ]:
queue = priority.head(DAILY_CAPACITY)[
    ["rank", "name_customer", "total_open_amount", "days_late_pred",
     "days_late_lower", "days_late_upper", "expected_value", "action"]].copy()
queue["p_late"] = [round(_p_late(l, u), 2) for l, u in
                   zip(priority.head(DAILY_CAPACITY).days_late_lower,
                       priority.head(DAILY_CAPACITY).days_late_upper)]
print("TODAY'S COLLECTION QUEUE\n")
print(queue.round(2).to_string(index=False))

That table is the actual output of the project. Not a prediction — a **work order**, ordered by
expected return, sized to what the team can do today.

## 14. Does the ranking beat the obvious alternatives?

The engine is only worth its complexity if it beats what a collections manager would do without
it. Two natural baselines:

- **Random** — call 20 invoices at random.
- **By amount** — call the 20 largest invoices, the standard heuristic.

In [ ]:
rng = np.random.default_rng(42)
random_ev   = priority.iloc[rng.choice(len(priority), DAILY_CAPACITY, replace=False)].expected_value.sum()
by_amount_ev = priority.nlargest(DAILY_CAPACITY, "total_open_amount").expected_value.sum()
engine_ev   = priority.nlargest(DAILY_CAPACITY, "expected_value").expected_value.sum()

strat = pd.Series({"Random": random_ev, "By amount": by_amount_ev, "Decision engine": engine_ev})
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(strat.index, strat.values, color=["tab:gray", "tab:orange", "tab:blue"])
ax.set_ylabel("expected value of 20 calls ($)")
ax.set_title("Daily expected value by prioritisation strategy")
for i, v in enumerate(strat.values):
    ax.text(i, v, f"${v:,.0f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

print(strat.round(2).to_string())
print(f"\nengine vs by-amount: {(engine_ev/by_amount_ev - 1)*100:+.1f}%")
print(f"annualised gap (250 working days): ${(engine_ev - by_amount_ev) * 250:,.0f}")

**Read the annualised figure with care.** It is the value of the engine's ranking *under its own
assumptions* — the same `P(responds)`, `d_accel` and `r_daily` the engine uses to rank. It is a
self-consistent comparison of orderings, **not** a measured business outcome. Validating it for
real requires a holdout experiment where some invoices are deliberately not called.

Stating that limitation plainly is worth more in a portfolio than a bigger number.

## 15. Where this goes next

Recap of the full path:

```
dataset.csv
   -> parse dates, split open / closed
   -> days_late = clear_date - due_in_date          (target, closed only)
   -> point-in-time customer timeline               (leakage fix)
   -> as-of join at posting_date -> 13 features
   -> chronological split (train on past)
   -> 3 GBMs: mean + P10 + P90
   -> predictions + 80% interval on open invoices
   -> P(late), P(responds) -> expected value
   -> tier assignment -> EV ranking -> top 20 = today's calls
```

**Open weaknesses, stated honestly:**

1. `P(responds)` and `DAYS_ACCELERATED` are assumptions, never validated against outcomes.
2. Cold-start customers are the weakest segment and have no dedicated handling.
3. The model under-predicts extreme lateness; the P90 bound partly compensates.
4. Open invoices are predicted late at a higher rate (~57%) than the closed base rate (~42%) —
   likely genuine Feb–May 2020 distribution shift, and a monitoring concern rather than a bug.

**The Databricks migration** turns the point-in-time timeline into a managed feature table with
proper as-of lookups, tracks these runs in MLflow, and adds the delayed-label join that would let
weakness (1) finally be measured: log every prediction, wait for the invoice to clear, join the
outcome back, and compare expected against actual.